# EXP027: DQN の項目選択が theta にどれだけ依存するか

学習済み DQN を用い、状態 `[theta]` を [-4, 4] の細かいグリッドで振ったときの

- DQN の argmax 項目（履歴マスクなし = そのthetaでの第一選択）と Q 上位項目
- 同じ theta での Fisher 情報量（FI）最大項目（= MFI の選択）

を対比する。DQN の argmax が theta を変えてもほとんど動かなければ、方策が状態にほぼ
不感で少数項目に張り付いている（＝使用項目バリエーションが狭い直接原因）ことを示す。

- 履歴マスクは適用しない。目的は「そのthetaで方策が最も好む項目」の theta 応答を見ること。
- `DATASET_KEY` と `GAMMA` で real / sim・推定法・割引率を切り替える。


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for root in [cwd, *cwd.parents]:
        if (root / "data").is_dir() and (root / "EXP027").is_dir():
            return root
    raise FileNotFoundError("Run this notebook inside the repository.")


ROOT = find_project_root()
MODEL_DIR = ROOT / "EXP027" / "models"
RESULTS_DIR = ROOT / "EXP027" / "results"
device = torch.device("cpu")

# Network hyperparameters (aligned with EXP027 DQN training).
INPUT_SIZE = 1
FIRST_HIDDEN = 50
SECOND_HIDDEN = 30
DROPOUT_RATE = 0.0

# Each entry maps a key to its item bank and a gamma -> model-path builder.
DATASETS = {
    "real_LNIRT_MLE": {
        "bank": ROOT / "data" / "LNIRT_CredentialForm1" / "real item bank.csv",
        "model": lambda g: MODEL_DIR
        / f"dqn_real_LNIRT_CredentialForm1_MLE_gamma_{g}.pt",
    },
    "sim_uncor_MLE": {
        "bank": ROOT / "data" / "uncorrelated_banks" / "item_bank_uncor_1.csv",
        "model": lambda g: MODEL_DIR / f"dqn_MLE_uncor_1_gamma_{g}.pt",
    },
    "sim_uncor_EAP_unif": {
        "bank": ROOT / "data" / "uncorrelated_banks" / "item_bank_uncor_1.csv",
        "model": lambda g: MODEL_DIR / f"dqn_EAP_unif_uncor_1_gamma_{g}.pt",
    },
}

# --- analysis configuration -------------------------------------------------
DATASET_KEY = "real_LNIRT_MLE"
GAMMA = "0.1"
THETA_GRID = np.round(np.arange(-4.0, 4.0 + 1e-9, 0.05), 4)
TOP_K = 5

TAG = f"{DATASET_KEY}_gamma_{GAMMA}"
print(f"Project root : {ROOT}")
print(f"Dataset key  : {DATASET_KEY}")
print(f"Gamma        : {GAMMA}")
print(f"Theta grid   : {THETA_GRID[0]} .. {THETA_GRID[-1]} (n={len(THETA_GRID)})")
print(f"Top-K        : {TOP_K}")


In [ ]:
class Net(nn.Module):
    def __init__(self, input_size, first_hidden, second_hidden, action_space, dropout_rate):
        super().__init__()
        self.fc1 = nn.Linear(input_size, first_hidden)
        self.fc2 = nn.Linear(first_hidden, second_hidden)
        self.out = nn.Linear(second_hidden, action_space)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = F.relu(self.dropout(self.fc1(x)))
        x = F.relu(self.dropout(self.fc2(x)))
        return self.out(x)


def FI(item_bank: np.ndarray, theta: float, D: float = 1.0) -> np.ndarray:
    a = item_bank[:, 0]
    b = item_bank[:, 1]
    c = item_bank[:, 2]
    return (
        D**2
        * a**2
        * (1 - c)
        / (c + np.exp(D * a * (theta - b)))
        / (1 + np.exp(-D * a * (theta - b))) ** 2
    )


spec = DATASETS[DATASET_KEY]
item_bank = pd.read_csv(spec["bank"])[["a", "b", "c"]].to_numpy(dtype=float)
action_space = len(item_bank)

model_path = spec["model"](GAMMA)
model = Net(INPUT_SIZE, FIRST_HIDDEN, SECOND_HIDDEN, action_space, DROPOUT_RATE).to(device)
model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
model.eval()

print(f"item bank : {item_bank.shape}  ({spec['bank'].name})")
print(f"model     : {model_path.name}")


In [ ]:
# Sweep theta; for each theta record the DQN argmax item and the FI-optimal
# (MFI) item, plus the top-K sets for each.
def q_values(theta: float) -> np.ndarray:
    state = torch.tensor([[theta]], dtype=torch.float32, device=device)
    with torch.no_grad():
        return model(state).cpu().numpy().flatten()


q_matrix = np.zeros((len(THETA_GRID), action_space))
rows = []
for i, theta in enumerate(THETA_GRID):
    q = q_values(theta)
    q_matrix[i] = q
    fi = FI(item_bank, theta)

    dqn_arg = int(q.argmax())
    fi_arg = int(fi.argmax())
    dqn_topk = np.argsort(q)[-TOP_K:][::-1]
    fi_topk = np.argsort(fi)[-TOP_K:][::-1]

    rows.append(
        {
            "theta": theta,
            "dqn_item": dqn_arg + 1,
            "dqn_b": item_bank[dqn_arg, 1],
            "dqn_a": item_bank[dqn_arg, 0],
            "dqn_q": q[dqn_arg],
            "fi_item": fi_arg + 1,
            "fi_b": item_bank[fi_arg, 1],
            "fi_a": item_bank[fi_arg, 0],
            "fi_value": fi[fi_arg],
            "argmax_agree": int(dqn_arg == fi_arg),
            "dqn_in_fi_topk": int(dqn_arg in fi_topk),
            "dqn_topk": ";".join(str(x + 1) for x in dqn_topk),
            "fi_topk": ";".join(str(x + 1) for x in fi_topk),
        }
    )

sweep = pd.DataFrame(rows)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
csv_path = RESULTS_DIR / f"analysis_q_argmax_vs_theta_{TAG}.csv"
sweep.to_csv(csv_path, index=False)

# State-sensitivity summary.
n_dqn_distinct = sweep["dqn_item"].nunique()
n_fi_distinct = sweep["fi_item"].nunique()
dqn_topk_union = len(set().union(*(s.split(";") for s in sweep["dqn_topk"])))
fi_topk_union = len(set().union(*(s.split(";") for s in sweep["fi_topk"])))

print(f"Saved: {csv_path}")
print()
print(f"theta grid points               : {len(THETA_GRID)}")
print(f"distinct DQN argmax items        : {n_dqn_distinct}")
print(f"distinct FI (MFI) argmax items   : {n_fi_distinct}")
print(f"distinct items in DQN top-{TOP_K} union : {dqn_topk_union}")
print(f"distinct items in FI  top-{TOP_K} union : {fi_topk_union}")
print(f"argmax agreement (DQN == MFI)    : {sweep['argmax_agree'].mean():.3f}")
print(f"DQN argmax within FI top-{TOP_K}       : {sweep['dqn_in_fi_topk'].mean():.3f}")
print()
print("DQN argmax item counts by theta region:")
print(sweep.groupby("dqn_item").size().sort_values(ascending=False).head(10))


In [ ]:
# Plot 1: which item is selected (argmax) as a function of theta, DQN vs MFI.
# Plot 2: difficulty b of the selected item vs theta (should track theta if the
#         policy adapts). A flat DQN curve indicates a state-insensitive policy.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(sweep["theta"], sweep["fi_item"], color="steelblue", lw=1.6,
             label="MFI (FI argmax)")
axes[0].plot(sweep["theta"], sweep["dqn_item"], color="tomato", lw=1.6,
             label="DQN argmax")
axes[0].set_title(f"Selected item ID vs theta  [{TAG}]")
axes[0].set_xlabel("theta")
axes[0].set_ylabel("item ID")
axes[0].grid(True, ls="--", alpha=0.4)
axes[0].legend()

axes[1].plot(sweep["theta"], sweep["fi_b"], color="steelblue", lw=1.6,
             label="MFI selected b")
axes[1].plot(sweep["theta"], sweep["dqn_b"], color="tomato", lw=1.6,
             label="DQN selected b")
axes[1].plot(sweep["theta"], sweep["theta"], color="gray", ls=":", lw=1.0,
             label="b = theta")
axes[1].set_title(f"Difficulty b of selected item vs theta  [{TAG}]")
axes[1].set_xlabel("theta")
axes[1].set_ylabel("selected item b")
axes[1].grid(True, ls="--", alpha=0.4)
axes[1].legend()

fig.tight_layout()
plot_path = RESULTS_DIR / f"analysis_q_argmax_vs_theta_{TAG}.png"
fig.savefig(plot_path, dpi=150)
plt.show()
print(f"Saved: {plot_path}")


In [ ]:
# Plot 3: Q(theta) curves for every item that is ever the DQN argmax on the
# grid. If only a few curves dominate everywhere and rarely cross, the greedy
# policy is state-insensitive and collapses onto a small item set.
ever_argmax = sorted(sweep["dqn_item"].unique())
item_indices = [i - 1 for i in ever_argmax]

fig, ax = plt.subplots(figsize=(12, 6))
cmap = plt.cm.tab20(np.linspace(0, 1, max(len(item_indices), 1)))
for color, item_idx in zip(cmap, item_indices):
    ax.plot(THETA_GRID, q_matrix[:, item_idx], lw=1.3, color=color,
            label=f"item {item_idx + 1} (b={item_bank[item_idx, 1]:.2f})")

ax.set_title(f"Q(theta) for items ever chosen by DQN argmax  [{TAG}]")
ax.set_xlabel("theta")
ax.set_ylabel("Q value")
ax.grid(True, ls="--", alpha=0.4)
if len(item_indices) <= 20:
    ax.legend(fontsize=8, ncol=2, loc="best")

fig.tight_layout()
qcurve_path = RESULTS_DIR / f"analysis_q_argmax_qcurves_{TAG}.png"
fig.savefig(qcurve_path, dpi=150)
plt.show()
print(f"items ever DQN-argmax on the grid: {ever_argmax}")
print(f"Saved: {qcurve_path}")
